In [1]:
# !pip install openrouter requests

In [2]:
# !pip install openai

In [3]:
import os
import json
import asyncio
import glob
from openai import AsyncOpenAI
from IPython.display import display, JSON

In [4]:

OPENROUTER_API_KEY = "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" 
MODELS = [
    # "anthropic/claude-opus-4.8"
    "qwen/qwen3.7-plus"
]

In [5]:

# prompt for metadata extraction
SYSTEM_PROMPT = """You are a metadata extraction system for artifacts documented on GitHub.

Your task is to read the provided artifact document and extract all useful metadata that would help someone understand, index, search, reproduce, or reuse the artifact.

The artifact document is a GitHub README file in Markdown format. Use its structure (headers, code blocks, badges, lists) to help you locate and delimit evidence spans and populate evidence_location — for example, citing a Markdown header as the location, or treating a fenced code block as a single coherent span. This is a structural note only: it does not imply a fixed list of expected fields, and you should still identify whatever metadata is available in this specific document, not a generic README template.

Use only the information explicitly stated in the document. Do not use outside knowledge. Do not infer facts that are not directly supported by the text.

You may decide by yourself which metadata fields are relevant and how to name them, based entirely on what this specific document contains. Do not restrict yourself to any fixed list of field types, and do not assume the artifact belongs to any particular kind of project (e.g. software library, machine learning model, dataset, research paper, hardware project, documentation site, etc.) unless the document itself establishes that. Name each field using whatever term best describes what you found. Prefer short, snake_case field names (e.g. "programming_language", not "the language used").

For each metadata item, return:
- field: a short snake_case name for the metadata type you identified
- value: the extracted value, as close to verbatim as possible
- normalized_value: apply ONLY these normalizations, nothing else:
  - trim surrounding whitespace/punctuation
  - collapse internal whitespace
  - do not shorten, expand, or abbreviate names (e.g. keep "MIT License" as "MIT License", not "MIT"; keep "Node.js 18" as-is, not "Node 18")
  If no normalization applies, repeat value exactly.
- evidence: a text span copied VERBATIM (character-for-character) from the artifact text, including original Markdown syntax (e.g. keep "## Installation" as-is, keep code fence contents as-is). Do not paraphrase, reflow, merge non-adjacent sentences, or fix typos in the evidence field. This field will be programmatically checked as a substring of the source document.
- evidence_location: line number, Markdown section/header title, or "unknown" if not available
- confidence: one of "high", "medium", "low", using these criteria:
  - high: the value is named explicitly and unambiguously, with no plausible alternative reading
  - medium: the value is supported but requires minor interpretation (e.g. resolving an acronym used elsewhere in the doc, or a value split across adjacent sentences)
  - low: the text is genuinely ambiguous between two or more readings, or the evidence is indirect/implicit
- reason: a short explanation of why this value is useful metadata

Important rules:
1. Extract only metadata supported by evidence in the artifact text.
2. If a possible metadata value is ambiguous, include it with confidence "low" and explain the ambiguity in "reason".
3. Do not output generic concepts unless the document names a concrete value.
4. Do not invent missing metadata.
5. If the same underlying fact could be tagged with more than one field name, choose the single field name you judge most precise, and do not emit duplicate items for the same evidence span under multiple field names.
6. If no useful metadata is found, return an empty list.
7. Output ONLY the JSON object below. No markdown code fences, no preamble, no explanation outside the JSON structure.

Return only valid JSON in the following format:

{
  "artifact_id": "<ARTIFACT_ID>",
  "metadata": [
    {
      "field": "...",
      "value": "...",
      "normalized_value": "...",
      "evidence": "...",
      "evidence_location": "...",
      "confidence": "high | medium | low",
      "reason": "..."
    }
  ]
}"""

In [6]:
folder_path = r"E:\TUB\WS25-26\fiz\new_task\readme_files"
md_files = glob.glob(os.path.join(folder_path, "*.md"))
documents = {}
for file_path in md_files:
    with open(file_path, "r", encoding="utf-8") as f:
        doc_name = os.path.basename(file_path)
        documents[doc_name] = f.read()
print(f"load {len(documents)} files:{list(documents.keys())}")

load 50 files:['000linlin__ConsisLoRA.md', '01yzzyu__wikiautogen.md', '0russwest0__Agent-R1.md', '0xnu__expectile-based-framework.md', '1202kbs__AYT.md', '1230young__bizgen.md', '13ari__DRISHTIKON.md', '1712n__eltex.md', '1akaman__EmoAgent.md', '1jsingh__negtome.md', '1wrx1__GIA.md', '1x-technologies__1xgpt.md', '2020-qqtcg__SRGen.md', '3145tttt__GAS.md', '3587jjh__lsrna.md', '360CVGroup__EVTAR.md', '360CVGroup__FancyVideo.md', '360CVGroup__FG-CLIP.md', '3dlg-hcvc__NuiScene.md', '3DTopia__4DNeX.md', '3DTopia__GenDoP.md', '3DTopia__GPTEval3D.md', '3DTopia__ThemeStation.md', '41xu__DEMO.md', '8421BCD__ReasonRank.md', 'AadSah__kyvo.md', 'aakaran__reasoning-with-sampling.md', 'aaronhuang-778__billm.md', 'aaronserianni__attention-iou.md', 'AaronZ345__ISDrama.md', 'AaronZ345__VersBand.md', 'aashiqmuhamed__DynamicSAEGuardrails.md', 'aashish2000__RONA.md', 'ab9mamun__AIMI.md', 'abdelfattah-lab__xKV.md', 'abdo-eldesokey__mind-the-glitch.md', 'abenechehab__AdaPTS.md', 'abenechehab__nll_to_po.md'

In [7]:
# ---------- 异步提取函数 ----------
async def extract_metadata(client, model, doc_content, doc_name="unknown"):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Please extract metadata from the following GitHub README document. The artifact_id should be inferred from the project name (use the repository name if available, or the title).\n\n---\n{doc_content}\n---"}
    ]
    try:
        completion = await client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=0,
            # 若模型支持 reasoning，可取消注释下一行
            # extra_body={"reasoning": {"effort": "high"}}
        )
        raw = completion.choices[0].message.content
        # 清理 Markdown 代码块
        json_str = raw.strip()
        if json_str.startswith("```json"):
            json_str = json_str.split("```json")[1].split("```")[0].strip()
        elif json_str.startswith("```"):
            json_str = json_str.split("```")[1].split("```")[0].strip()
        result = json.loads(json_str)
        return {"model": model, "doc": doc_name, "success": True, "result": result}
    except Exception as e:
        return {"model": model, "doc": doc_name, "success": False, "error": str(e), "raw_response": raw if 'raw' in locals() else None}

In [ ]:
async def main():
    client = AsyncOpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=OPENROUTER_API_KEY,
    )
    all_results = []
    for doc_name, content in documents.items():
        tasks = [extract_metadata(client, model, content, doc_name) for model in MODELS]
        results = await asyncio.gather(*tasks)
        all_results.append({"document": doc_name, "results": results})
    return all_results

results = await main()

In [ ]:
for doc_group in results:
    doc_name = doc_group['document'].replace('.md', '')
    with open(f"{doc_name}_metadata_qwen.json", "w", encoding="utf-8") as f:
        json.dump(doc_group['results'], f, ensure_ascii=False, indent=2)